# Lab 02: The Agent Loop & Design Patterns

Kiel University · Agentic AI (infAgAI-01a) · Winter 2026

**Learning objectives.** After this lab you can:

- build the **minimal agent loop** (reason, act, observe, decide) around `llm.chat`, with the message list as the agent's entire working state,
- give the model two **tools** as JSON schemas and execute its tool calls in your own Python code, turning every error into an observation,
- **instrument** the loop and measure how resending the whole history makes the cumulative input grow quadratically,
- **break** the loop on purpose and read the trajectories: runaway loop, context overflow, derailment,
- name the **first-line defences** against each failure mode, and try one of them if you have time.

*Requests are limited. Your key allows 200 model requests per hour; one pass through Parts A to E needs about 70. Re-run the long cells of Part E only when you need to.*

**Report tasks and hand-in.** Tasks marked 📝 are the report tasks R1 and R2. They come without solutions, and your answers go **directly into this notebook**: each report task is followed by an answer cell marked ✍️ that you fill in. Paste the numbers, log lines and trajectories a task asks for into the ✍️ cell as text: the graders do not see cell outputs. There is nothing to upload. The notebook is collected automatically from your folder on the lab server on **Thursday of the week after the lab, at 18:00** (eight days after your lab session); whatever the notebook contains at that moment counts. Save it before then.

## Theory recap: one loop, one growing message list

Everything in this recap is from Session 02; the code walkthrough *mini_agent.py, line by line* explains the lecture's program line by line, and this lab rebuilds that program with the lab's client. The three ideas you need at your fingertips: **the model proposes, the runtime executes**; **the message list is the state**; **the worst failures show only in the trajectory**. Open the recap if any of these is not clear, and come back to it for the failure table (Part E) and the cost formula (Part D, R1).

<details>
<summary><b>Click here for the full recap (loop, state, stop conditions, failure modes, quadratic cost, patterns)</b></summary>

**The loop is the agent.** You built it line by line in the lab on the minimal agent: one iteration is reason (the whole message list goes to the model), act (the runtime executes the tool call), observe (the result joins the list as a tool message) and decide (a tool call means loop again, no tool call ends the run). **The model proposes; the runtime executes.** Tool execution, permissions and budgets are your code and your responsibility.

**State.** The model keeps nothing between calls. Three nested places hold the state of a run: the **context** (the message list, everything the model sees, sent in full each time), the **runtime** (your loop: message list, step and token counters, stop logic) and the **environment** (the world outside the agent, where side effects persist).

**Four ways a loop should end.** *No further action*: the model emits no tool call, so there is nothing left to execute. *Step cap*: a hard iteration limit, the last defence when the model never converges. *Budget cap*: stop on cumulative tokens, money or time. *Human interrupt*: approval gates and a kill switch. The minimal agent has only the first two; Part F suggests how to add the third. None of these exits means success, not even the first: *no further action* says only that the model stopped calling tools. Its last message can be a complete answer, a refusal, or an answer cut off at the token limit. The loop does not read that message; it only checks for a tool call, so it ends the same way in all three cases, although a human or a model reading the text would tell them apart at once. So the runtime should check `finish_reason` (`length` means cut off). A refusal it cannot catch that way, because it ends with `stop` like a real answer. Whether the task is solved is judged outside the loop: in this lab by you, reading the final answer and the trajectory; in a real system by an evaluation step such as a test or a judge model.

**Three common failure modes.** None of them needs to raise an exception: each step can succeed, and the problem shows only in the **trajectory**, the sequence of steps in one run.

| Failure mode | What happens | Early symptom | First-line defence |
|---|---|---|---|
| Infinite loop | same call, same arguments, hoping for a different result | identical tool call repeated across steps | step cap; duplicate-call detector |
| Context overflow | the message list gets longer than the model's limit (the context window); the provider rejects the request, or the oldest messages are silently dropped (by the provider or by an agent framework) | latency and cost rise every step | truncate observations; summarise old turns |
| Derailment | locally reasonable steps drift away from the stated goal | actions stop serving the goal | re-inject goal; plan checkpoints; human review |

**The quadratic cost.** Let $I_0$ be the input of the first request (system prompt, tool schemas, task) and $\Delta$ what each iteration adds on average. The input of request $k$ is $I_0 + (k-1)\Delta$, so the sum over $n$ requests is $n \cdot I_0 + \Delta \cdot n(n-1)/2$. With $I_0 = \Delta = 2{,}000$ tokens and $n = 20$: 420,000 cumulative input tokens against a final context of 40,000, about ten times over. Cumulative output tokens grow only linearly. Five cost levers: prompt caching, observation hygiene, compaction, model routing, budget caps. Rule of thumb: shorten observations before shortening the loop.

**Four design patterns (Ng, 2024).** Planning, Tool Use, Reflection, Multi-Agent: each is a disciplined way to spend extra tokens for better behaviour, and none of them changes the loop. Tool Use is the pattern this lab's agent already implements.

</details>

## Part A: Setup

Same model and client as in the earlier labs: `llm.chat()` reaches the course model through `llm.py` in this folder. For most of you that is `qwen3-30b-a3b-instruct-2507` on the GWDG Academic Cloud; if your account was switched to the lab server's models, you get one of those instead. Either way, `import llm` prints one line that tells you which provider and model you are using (`llm: using <provider>, model <model>`). Both support tool calling. The lecture's `call_model` with its bounded retries is already built into `llm.chat`, so you call it directly. This lab also uses `pandas` and `matplotlib`.

The cell below imports everything and checks the connection.

In [ ]:
import json
import time

import pandas as pd
import matplotlib.pyplot as plt

import llm

MODEL = llm.MODEL  # name of the course model

try:
    resp = llm.chat([{"role": "user", "content": "Reply with exactly one word: ready"}], model=MODEL)
    print(f"Model '{MODEL}' answered:",
          resp["choices"][0]["message"]["content"].strip())
except Exception as e:
    print("Could not talk to the model:", e)
    print("Check the hint in the message above; without a saved key, run the Onboarding notebook first.")

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `import json` loads Python's module for JSON, the text format in which the model writes the arguments of a tool call; we need `json.loads` to turn such a text into a dictionary. `import time` gives us `time.time()`, the current time in seconds, for measuring how long a model call takes.
- `pandas` (abbreviated `pd`) is the table library, used for the log in Part D; `matplotlib.pyplot` (abbreviated `plt`) draws the plot there.
- `import llm` loads the file `llm.py` from this folder, the same small client as in the earlier labs. `llm.MODEL` is the name of the course model; we copy the name into `MODEL` so that every call can say `model=MODEL`.
- The `try` block sends one tiny message. If the call succeeds, the reply is printed. If anything goes wrong (no key, no network, overloaded server), Python jumps to `except Exception as e:`, where `e` holds the error, and prints a hint instead of stopping. `resp["choices"][0]["message"]["content"]` is the path to the reply text inside the response dictionary, as in the earlier labs.
- While loading, `llm.py` itself prints one line that names the provider and the model (only on the first import in a session). `llm.chat([...], model=MODEL)` receives a message list with a single user message. `print` gets two arguments separated by a comma and prints them with a space between; the first is an f-string that inserts `MODEL`, and `.strip()` removes spaces and line breaks at both ends of the reply.

</details>

## Part B: The tools: a tiny offline web

The agent needs a web to search. We simulate one: `data/search_corpus.json` holds nine short pages written for this lab: four on the EU AI Act, three on US AI policy (one of them links to the other two; you will meet it again in Part E), one noise page about pasta, and one deliberately gigantic page for Part E. The pages are simplified teaching material, not legal sources. Compared with the three-page web of the minimal-agent lab there are two tools now:

- `web_search(query)`: keyword search, returns the top three pages as 300-character snippets,
- `read_page(page_id)`: returns the full text of one page, which can be very long.

The cell below is given; read it, run it, and look at what a search result looks like. The model will see exactly this text as an observation.

<div style="text-align: center;">
<img src="images/fig01_two_tools.png" width="960" alt="The mini-web and the two tools web_search and read_page">
</div>

In [ ]:
with open("data/search_corpus.json", encoding="utf-8") as f:
    CORPUS = json.load(f)  # {page_id: {"title": ..., "content": ...}}

print(f"{len(CORPUS)} pages in the mini-web:")
for page_id, page in CORPUS.items():
    print(f"  {page_id:24} {page['title'][:60]}  ({len(page['content'])} characters)")

STOPWORDS = {"the", "and", "for", "what", "are", "does", "with", "that",
             "this", "from", "how", "into", "was", "were", "has", "have"}

def keywords(text):
    '''Lower-cased words of a text, punctuation stripped, stopwords and short words removed.'''
    words = [w.strip(".,?!:;\"'()[]").lower() for w in text.split()]
    return [w for w in words if len(w) > 2 and w not in STOPWORDS]

def web_search(query: str) -> str:
    '''Search the mini-web; return the top three hits as short snippets.'''
    scores = {}
    for page_id, page in CORPUS.items():
        page_words = keywords(page["title"] + " " + page["content"])
        scores[page_id] = sum(page_words.count(w) for w in keywords(query))
    top = [page_id for page_id in sorted(scores, key=scores.get, reverse=True)[:3]
           if scores[page_id] > 0]
    if not top:
        return "No results found."
    return "\n\n".join(f"[{page_id}] {CORPUS[page_id]['title']}\n{CORPUS[page_id]['content'][:300]} ..."
                       for page_id in top)

def read_page(page_id: str) -> str:
    '''Return the FULL text of one page (can be very long).'''
    if page_id not in CORPUS:
        return f"TOOL ERROR: unknown page_id '{page_id}'. Use web_search to find valid page ids."
    return CORPUS[page_id]["content"]

print()
print(web_search("EU AI Act obligations general-purpose models")[:600])

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `with open(...) as f:` opens the file and closes it automatically at the end of the indented block; `json.load(f)` turns the JSON text into Python objects: a dictionary from page id to a dictionary with `title` and `content`.
- `for page_id, page in CORPUS.items():` walks through the dictionary pair by pair: `.items()` yields each key together with its value, and the two loop variables receive them, `page_id` the key and `page` the inner dictionary. Looping over `CORPUS` alone would give only the keys.
- In the `print` line, `{page_id:24}` pads the id to 24 characters so that the titles line up; `page['title'][:60]` is a **slice**, the first 60 characters; `len(page['content'])` is the length of the page text. The keys use single quotes because the f-string itself is written with double quotes.
- `STOPWORDS` is a **set** (curly braces): a collection without order and without duplicates, good for the quick test `w not in STOPWORDS`.
- `def web_search(query: str) -> str:` carries **type hints**: the parameter should be a string, and the function returns a string. Python does not enforce them; they document the interface for the reader. The triple-quoted line under each `def` is the **docstring**, the function's own description.
- `keywords(text)` splits a text into words, strips punctuation from both ends of each word (`strip` with the characters to remove), lower-cases it, and drops short words and stopwords. Two list comprehensions, the compact loop form you know.
- `web_search` scores every page by counting, for each query word, how often it occurs in the page (`page_words.count(w)`); `sorted(scores, key=scores.get, reverse=True)[:3]` sorts the page ids by score, highest first, and keeps three; pages with score 0 are dropped, so a query with no matching word yields `"No results found."`. The result is one text with the page id in square brackets, the title and the first 300 characters of each hit: the model never receives Python objects, only this text.
- Four smaller constructs in `web_search`: `sum(... for w in keywords(query))` adds up the counts directly, a comprehension without square brackets (a **generator expression**); `key=scores.get` tells `sorted` to compare the page ids by their score, the value that `scores.get(page_id)` returns; `if not top:` is true for an empty list, because an empty list counts as false; `"\n\n".join(...)` glues the hits together with an empty line between them.
- `read_page` is the new second tool: the full content of one page. For an unknown id it returns an error **string** instead of raising an exception, so that the model can read it and correct itself.
- `page_id not in CORPUS` tests the **keys** of the dictionary. The last two lines print an empty line and then the first 600 characters of a sample search result (`[:600]`).

</details>

### Describing the tools to the model

`TOOLS` and `FUNCS` work exactly as in the minimal-agent lab; the cell is given. New is only the second entry, and one sentence in the first description that tells the model when to use it: snippets are samples, not full pages.

> **Key point.** The descriptions are instructions to the model: they are prompt text.

In [ ]:
TOOLS = [
    {"type": "function",
     "function": {
         "name": "web_search",
         "description": ("Search an offline copy of the web. Returns the top three matching "
                         "pages as [page_id], title and a 300-character snippet. "
                         "Snippets are samples, not full pages."),
         "parameters": {"type": "object",
                        "properties": {"query": {"type": "string",
                                                 "description": "keyword search query"}},
                        "required": ["query"]}}},
    {"type": "function",
     "function": {
         "name": "read_page",
         "description": "Read the FULL text of one page. Use a page_id returned by web_search.",
         "parameters": {"type": "object",
                        "properties": {"page_id": {"type": "string"}},
                        "required": ["page_id"]}}},
]

FUNCS = {"web_search": web_search, "read_page": read_page}  # name the model uses -> Python function

print("Tools the model will know about:", [t["function"]["name"] for t in TOOLS])

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The structure is the one you walked through level by level in the minimal-agent lab: a list with one dictionary per tool, each with `name`, `description` and a JSON Schema under `parameters`.
- The description of `web_search` is written as several string pieces inside parentheses (Python joins adjacent string literals). Its last sentence, "Snippets are samples, not full pages", exists for the model: without it, models tend to answer from snippets alone and never call `read_page`.
- `FUNCS` now maps two names to two functions. Each name must be spelled exactly as in `TOOLS`.
- In `FUNCS` the values are the functions themselves, written without parentheses: `web_search` names the function, `web_search(...)` would call it. The `print` line uses a list comprehension to collect the `name` of every tool in `TOOLS`.

</details>

## Part C: The minimal agent loop

You built `run_tool` and the loop line by line in the minimal-agent lab. Both come back here as the base for everything else, with three additions that this lab needs:

1. `run_tool` takes the truncation limit as a parameter, `obs_limit`, so that Part E can switch it off and on,
2. the loop is a function `run_agent(task, ...)` that **returns** the answer and the message list instead of printing them, so that we can run it many times and compare,
3. the loop writes one **log row per step** (tokens and latency of each request), because every later part of this lab measures what the loop does.

First `run_tool` with change 1, given. Compared with the minimal-agent lab, only two lines changed: the new parameter `obs_limit` in the first line, and the cut to `obs_limit` characters instead of a fixed 4000 in the `return`. Changes 2 and 3 are in the loop, `run_agent`, two code cells further down.

In [ ]:
def run_tool(call, obs_limit=4000):
    '''Run one requested call; errors become observations.'''
    try:
        fn = FUNCS[call["function"]["name"]]              # unknown tool -> KeyError
        args = json.loads(call["function"]["arguments"])  # invalid JSON -> error
        result = fn(**args)                               # bad arguments -> TypeError
    except Exception as e:                                # any error becomes text
        result = f"TOOL ERROR: {e!r}"
    return {"role": "tool", "tool_call_id": call["id"],
            "content": str(result)[:obs_limit]}           # str(): a tool may return a non-string; cut to obs_limit characters, not tokens

# Smoke test with a hand-made tool call, as the model would write it:
test_call = {"id": "call_0", "type": "function",
             "function": {"name": "web_search", "arguments": '{"query": "systemic risk"}'}}
print(run_tool(test_call)["content"][:200])

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- The body is the function from the minimal-agent lab: look the function up by name, parse the argument string with `json.loads`, call it with the arguments unpacked (`fn(**args)`), and turn any error into a `TOOL ERROR` text; `{e!r}` shows the error with its type.
- The return value is the tool message: the role `tool`, the `tool_call_id` that ties the result to the request it answers, and the result as text. `str(result)` makes sure it is a string before the slice cuts it.
- New is the parameter `obs_limit=4000`. The default value 4000 is the lecture's `[:4000]`; the slice `[:obs_limit]` now cuts the observation to whatever the caller passes. Part E passes a huge value to let a gigantic page through, and a small one to see what the cut destroys.
- The smoke test builds one tool call by hand, in the shape the model produces (`"arguments"` is a JSON **string**), and prints the first 200 characters of the resulting observation.

</details>

<div style="text-align: center;">
<img src="images/fig02_loop_flow.png" width="960" alt="Flow chart of run_agent: one round, three returns">
</div>

### The loop as a function, with a log

The loop below is the one you know, wrapped in a function. In the exercise you fill in only the new lines: the log row (which two token counts does the server report?), the exits (what does the function return in each case?) and the call to `run_tool` with the new parameter. Everything else is as in the minimal-agent lab.

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful research assistant working against a small offline copy of the web. "
    "Use web_search to find pages and read_page to read a page in full. "
    "When you have gathered enough evidence, give your final answer as plain text "
    "with page_id citations in square brackets."
)

def run_agent(task, max_steps=8, obs_limit=4000, verbose=True):
    '''The minimal agent loop: reason, act, observe, decide. Returns (answer, messages, log).'''
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": task},
    ]
    log = []                                                      # one row per step, for Part D
    for step in range(max_steps):                                 # stop: step cap
        t0 = time.time()
        resp = llm.chat(messages, model=MODEL, tools=TOOLS)       # REASON: the FULL history, every time
        msg = resp["choices"][0]["message"]
        messages.append(msg)                                      # state grows here
        calls = msg.get("tool_calls") or []                       # [] when the model did not call a tool
        log.append({"step": step,
                    "prompt_tokens": resp["usage"][___],      # what the model read: the context size
                    "completion_tokens": resp["usage"][___],  # what the model wrote
                    "latency_s": round(time.time() - t0, 1),
                    "action": calls[0]["function"]["name"] if calls else "answer"})  # log first call only

        if not calls:                                             # no further action: DECIDE
            if resp["choices"][0]["finish_reason"] == "length":
                if verbose:
                    print(f"[step {step}] answer cut off at the token limit")
                return ___, messages, log                     # no usable answer
            if verbose:
                print(f"[step {step}] final answer")
            return ___, messages, log                         # the answer text

        for call in calls:                                        # each requested call
            if verbose:
                print(f"[step {step}] {call['function']['name']}({call['function']['arguments']})")
            observation = run_tool(call, ___)                 # ACT, with the limit from the arguments
            messages.append(observation)                          # OBSERVE

    if verbose:                                                   # reached only if the loop ran out of steps
        print("Step budget exhausted, no answer.")
    return ___, messages, log

<details>
<summary><b>Click here for the solution</b></summary>

```python
SYSTEM_PROMPT = (
    "You are a careful research assistant working against a small offline copy of the web. "
    "Use web_search to find pages and read_page to read a page in full. "
    "When you have gathered enough evidence, give your final answer as plain text "
    "with page_id citations in square brackets."
)

def run_agent(task, max_steps=8, obs_limit=4000, verbose=True):
    '''The minimal agent loop: reason, act, observe, decide. Returns (answer, messages, log).'''
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": task},
    ]
    log = []                                                 # one row per step, for Part D
    for step in range(max_steps):                            # stop: step cap
        t0 = time.time()
        resp = llm.chat(messages, model=MODEL, tools=TOOLS)  # REASON: the FULL history, every time
        msg = resp["choices"][0]["message"]
        messages.append(msg)                                 # state grows here
        calls = msg.get("tool_calls") or []                  # [] when the model did not call a tool
        log.append({"step": step,
                    "prompt_tokens": resp["usage"]["prompt_tokens"],  # what the model read: the context size
                    "completion_tokens": resp["usage"]["completion_tokens"],  # what the model wrote
                    "latency_s": round(time.time() - t0, 1),
                    "action": calls[0]["function"]["name"] if calls else "answer"})  # log first call only

        if not calls:                                        # no further action: DECIDE
            if resp["choices"][0]["finish_reason"] == "length":
                if verbose:
                    print(f"[step {step}] answer cut off at the token limit")
                return None, messages, log                   # no usable answer
            if verbose:
                print(f"[step {step}] final answer")
            return msg["content"], messages, log             # the answer text

        for call in calls:                                   # each requested call
            if verbose:
                print(f"[step {step}] {call['function']['name']}({call['function']['arguments']})")
            observation = run_tool(call, obs_limit)          # ACT, with the limit from the arguments
            messages.append(observation)                     # OBSERVE

    if verbose:                                              # reached only if the loop ran out of steps
        print("Step budget exhausted, no answer.")
    return None, messages, log
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `SYSTEM_PROMPT` names both tools and tells the model how to end: plain text with page-id citations.
- `def run_agent(task, max_steps=8, obs_limit=4000, verbose=True):` wraps the loop you know in a function. `max_steps` is the step cap (the lecture's `MAX_STEPS`), `obs_limit` is passed on to `run_tool`, and `verbose=False` keeps a run silent (Part E prints only trajectories).
- `log = []` is our measuring instrument, not part of the agent: an empty list that receives one dictionary per step.
- `for step in range(max_steps):` runs the body at most `max_steps` times, with `step` counting 0, 1, 2, ... `llm.chat(messages, model=MODEL, tools=TOOLS)` sends the complete message list together with the tool schemas; `resp["choices"][0]["message"]` is the assistant message in the response, and it is appended to `messages` unchanged.
- `t0 = time.time()` notes the time before the request; `time.time() - t0` after it is the latency in seconds, rounded to one decimal with `round(..., 1)`.
- `calls = msg.get("tool_calls") or []`: the list of requested tool calls, or an empty list when the model wrote plain text (`or []` replaces `None` by `[]`), so that `if not calls`, `calls[0]` and `for call in calls` work in both cases.
- `msg.get("tool_calls")` reads a key that may be missing: `.get` returns `None` where `msg["tool_calls"]` would raise a `KeyError`.
- The log row records the step number, the two token counts the server reports in `resp["usage"]` (`prompt_tokens`: everything the model read in this request, which is the current context size in tokens; `completion_tokens`: what it wrote), the latency, and the action: `calls[0]["function"]["name"] if calls else "answer"` is a conditional expression that gives the name of the first requested tool, or the word `"answer"` when there is none.
- The exits are the ones you know, but with `return` instead of `print` and `break`: `if not calls:` is the no-further-action test; no further action plus `finish_reason == "length"` returns `None` (cut off, no usable answer); no further action otherwise returns `msg["content"]`, the answer text; the lines after the loop are reached only when all `max_steps` rounds passed without a return, and return `None` as well. Each `return` hands back three values at once (a tuple), which the caller unpacks into three names.
- `run_tool(call, obs_limit)` passes the limit from the function arguments on to the tool execution; the tool message is appended as before.
- `if verbose:` prints a progress line only when the caller did not switch it off. In `f"[step {step}] {call['function']['name']}(...)"` the dictionary keys use single quotes because the f-string itself is written with double quotes.

</details>

Run it on a real task. Expect a few steps: a search, one or two page reads, then an answer. Fill the gaps.

In [ ]:
task = ("Write a short, sourced report on the obligations the EU AI Act places on providers "
        "of general-purpose AI models. Cite page ids in square brackets.")

answer, transcript, log = run_agent(___)

print("\n=== ANSWER ===")
print(answer)
print(f"\nmessages in the final context: {len(___)}")

<details>
<summary><b>Click here for the solution</b></summary>

```python
task = ("Write a short, sourced report on the obligations the EU AI Act places on providers "
        "of general-purpose AI models. Cite page ids in square brackets.")

answer, transcript, log = run_agent(task)

print("\n=== ANSWER ===")
print(answer)
print(f"\nmessages in the final context: {len(transcript)}")
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `task` is the user's request, one string written over two lines inside parentheses.
- `answer, transcript, log = run_agent(task)` calls the loop with the default settings and unpacks the three returned values: the answer text (or `None`), the complete message list (we call it `transcript` here, because it is the record of the whole run) and the log table for Part D.
- `len(transcript)` is the number of messages in the final context: system prompt, task, one assistant message per step and one tool message per tool call.
- `\n` at the start of a printed string is a line break, so the output begins with an empty line.

</details>

❓ **Q:** What does it mean that the model is stateless, and where does the agent's state live instead?
<details><summary>Click for answer</summary>

Between requests the model retains nothing: each request is processed from scratch, from the context it is given to the tokens it writes. The agent's working state lives in three nested places outside the model: the context (the message list, sent in full each time), the runtime (the loop with its counters and stop logic) and the environment (the world, where side effects such as a sent e-mail persist). The "agent" is the loop plus its data, not the model.

</details>

## Part D: Instrumentation: watching the loop work

The loop has written one log row per step. Turn the log into a table and look at three numbers per step: `prompt_tokens`, the size of the context the model read in that request; its running total, the cumulative input the server has processed so far (with a commercial API, this is what you are billed for); and the running total of the output. One thing to know before you look: a provider may reuse the unchanged start of the previous request (**prompt caching**), which makes the call faster and, with paid providers, cheaper; the token count does not show that saving. Fill the gaps.

In [ ]:
log_df = pd.DataFrame(log)
log_df["cumulative_input"] = log_df["prompt_tokens"].___()  # running total (cumulative sum) of input tokens
log_df["cumulative_output"] = log_df["completion_tokens"].cumsum()  # running total (cumulative sum) of output tokens
display(log_df)

log_df.plot(x="step", y=["prompt_tokens", ___, "cumulative_output"],
            marker="o", xticks=log_df["step"], ylabel="tokens",
            title="Context size per request, cumulative input and cumulative output")
plt.show()

<details>
<summary><b>Click here for the solution</b></summary>

```python
log_df = pd.DataFrame(log)
log_df["cumulative_input"] = log_df["prompt_tokens"].cumsum()  # running total (cumulative sum) of input tokens
log_df["cumulative_output"] = log_df["completion_tokens"].cumsum()  # running total (cumulative sum) of output tokens
display(log_df)

log_df.plot(x="step", y=["prompt_tokens", "cumulative_input", "cumulative_output"],
            marker="o", xticks=log_df["step"], ylabel="tokens",
            title="Context size per request, cumulative input and cumulative output")
plt.show()
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `pd.DataFrame(log)` turns the list of dictionaries into a table: one row per dictionary (one per step), one column per key. `log_df` is a DataFrame, the pandas table type from the earlier labs.
- `log_df["prompt_tokens"].cumsum()` computes the **cumulative sum** of a column: entry $k$ is the sum of entries 0 to $k$. Because every request re-reads the whole context, this running total is what the run has cost in input tokens so far. Writing `log_df["cumulative_input"] = ...` adds the result as a new column. The next line does the same for the output tokens: each of them is counted once, when it is written; when later requests read it again, that shows up in the input.
- `display(log_df)` shows the table (Jupyter would show it automatically only if it were the last line of the cell).
- `log_df.plot(x="step", y=[...], marker="o", ...)` is pandas' shortcut for a line plot: the `step` column on the horizontal axis, one line per column named in `y`, a dot at every data point (`marker="o"`), one tick per step on the horizontal axis (`xticks=log_df["step"]`, otherwise pandas would also label half steps), a label for the vertical axis and a title. The call is written over three lines inside its parentheses. `plt.show()` displays the figure; `plt` is matplotlib, imported in Part A.

</details>

**Reading the plot.** The context, `prompt_tokens`, grows step by step: each step adds the model's message and the tool results. Because every request re-reads all of it, the cumulative input pulls away from the context size, faster with every step, while the cumulative output stays close to the bottom of the plot. That widening gap is the lecture's quadratic cost of re-reading: tokens the model has already read, read again in every later request. With the four or five points of one short run the bend is slight; R1 extrapolates it to 20 steps. Prompt caching reduces this cost for loop-shaped workloads: most of each request is a prefix the model has already read.

<div style="text-align: center;">
<img src="images/fig03_quadratic_tax.png" width="960" alt="The quadratic cost: every request re-reads the whole message list">
</div>

❓ **Q:** Why does the cumulative output (the sum of `completion_tokens` over all steps) grow only linearly, while the cumulative input grows quadratically?
<details><summary>Click for answer</summary>

Each message or tool call is generated exactly once, so the output of a run is the sum of the per-step generations: linear in the number of steps. Every generated or observed token, however, is re-read as input in all later requests, so early tokens are paid for again and again. Summing the re-reads gives $n \cdot I_0 + \Delta \cdot n(n-1)/2$, a quadratic term.

</details>

> **📝 Report task R1:** Using your log from Part D: (a) read $I_0$ off the first row (the `prompt_tokens` of step 0) and estimate the average $\Delta$ as (last `prompt_tokens` − first `prompt_tokens`) / (number of requests − 1); (b) use the lecture's formula $n \cdot I_0 + \Delta \cdot n(n-1)/2$ to predict the cumulative input of a 20-step run, and compare it with the size of the final context, $I_0 + 19\Delta$: how many times over has each token of the final context been paid for? (c) In one or two sentences: is $\Delta$ really constant in your log, and which kind of step adds most?
>
> *No solution is provided. Write your answer in the ✍️ cell below and paste the rows of your log there as text: the graders do not see cell outputs.*

✍️ **Your answer to R1.** Replace this line with your answer (Markdown, (a) to (c) as in the task, with the rows of your log pasted as text).

## Part E: Breaking the loop

The lecture announced it: *build this loop, then break it on purpose.* Model runs are random, and a well-behaved run is a result too: if an experiment does not misbehave, run it once more, then describe what you saw.

> **Key point.** Runaway loop and derailment raise no exception: every single step succeeds, and the problem exists only in the trajectory. Context overflow can be loud (the provider rejects the request) or silent (the provider or a framework drops the oldest messages without telling you).

<div style="text-align: center;">
<img src="images/fig04_failure_modes.png" width="960" alt="Three ways to break the loop: runaway loop, context overflow, derailment">
</div>

The helper below lists the tool calls of a run as short strings, so that we can count repeats and print trajectories. It is given; read the explanation.

In [ ]:
def tool_calls_of(messages):
    '''All tool calls of a run as strings like 'web_search({"query": "..."})', in order.'''
    calls = []
    for m in messages:
        for call in (m.get("tool_calls") or []):
            calls.append(f"{call['function']['name']}({call['function']['arguments']})")
    return calls

print(" -> ".join(tool_calls_of(transcript)))  # the trajectory of the Part C run

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `for m in messages:` goes through every message of a run. `m.get("tool_calls") or []` is the list of tool calls in that message, or an empty list for messages without any (system, user, tool and plain-text assistant messages), so the inner loop simply does nothing for them.
- For each call, an f-string builds `name(arguments)` from the two fields of the call; the arguments are kept as the JSON string the model wrote. `calls.append(...)` adds it to the list, in the order of the run.
- `" -> ".join(tool_calls_of(transcript))` prints the trajectory of the Part C run as one line of arrows. `join` is a method of every string: the string it is called on is the separator, and its argument is a sequence of strings. It returns one new string with the separator between each two elements: `" -> ".join(["a", "b", "c"])` gives `"a -> b -> c"`, the same as `"a" + " -> " + "b" + " -> " + "c"`. The method belongs to the separator, not to the list, so that it works for any sequence of strings: a list, a tuple or a generator.

</details>

### Experiment 1: the runaway loop and the step cap

We give the agent a task whose answer does **not** exist on the web, insist that it is there, and loosen the step cap. Against the plain mini-web, a well-behaved model still gives up after a few empty searches. On the real web, though, a search for something that does not exist rarely ends with "No results found.": every search offers another link that looks promising. The first cell below simulates such a search. Watch what the agent does: it may repeat a search word for word, follow one link after another, or give up. Fill the gaps: a generous cap, and the set that counts how many calls were exact repeats.

In [ ]:
# On the real web, a search for something that does not exist rarely says
# "No results found.": it usually returns some page that looks related.
# This search simulates that: every hit points to a further page, never to
# the answer. Experiment 1 swaps it in for web_search and puts the real
# search back afterwards.
part = 0

def never_ending_search(query: str) -> str:
    '''Return one promising hit that points to a further page, never the answer.'''
    global part
    part += 1
    return (f"[kiel_practical_part_{part}] Agentic AI practical sessions, Kiel: assessment, part {part}\n"
            f"The final mark for the practical sessions is explained in part {part + 1} of this guide. "
            f"Search for 'Agentic AI Kiel assessment part {part + 1}' ...")

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `never_ending_search` has the same signature as `web_search`: it takes a query string and returns text. The model cannot tell the two tools apart; only the results differ.
- `part` is defined outside the function, by `part = 0`. `global part` does not define it again; it tells Python that `part` inside the function means that outer variable. Without it, the assignment in `part += 1` would make `part` a new local variable without a value, and Python would raise an `UnboundLocalError`.
- The query is ignored: whatever the model asks, the result is the next part. Each hit looks relevant (right course, right city, "assessment") and promises the answer in the following part, so searching again always seems the sensible next step.
- The return value imitates the format of `web_search`: `[page_id] title`, then a snippet. Its last sentence even suggests the next query, as real pages do with their links.
- The experiment cell sets `part = 0` before each run, so every run starts again at part 1.

</details>

In [ ]:
# Nothing on the web answers this task, but every hit points to a further page.
doomed_task = ("What final mark does one receive for the Agentic AI practical "
               "sessions in Kiel? Answer only from web sources; do not guess. "
               "The answer is definitely on the web: follow every link and keep "
               "searching until you find the mark. Never give up.")

part = 0                                                        # start again at part 1
FUNCS["web_search"] = never_ending_search                        # Experiment 1 only
try:
    answer, transcript, log = run_agent(doomed_task, max_steps=___)  # generous cap, e.g. 10
finally:
    FUNCS["web_search"] = web_search                            # always restore the real search

calls = tool_calls_of(transcript)
repeats = len(calls) - len(___)                          # a set keeps each distinct call once
print(f"\n{len(calls)} tool calls, {repeats} of them exact repeats")
print("answer:", answer)

<details>
<summary><b>Click here for the solution</b></summary>

```python
# Nothing on the web answers this task, but every hit points to a further page.
doomed_task = ("What final mark does one receive for the Agentic AI practical "
               "sessions in Kiel? Answer only from web sources; do not guess. "
               "The answer is definitely on the web: follow every link and keep "
               "searching until you find the mark. Never give up.")

part = 0                                                        # start again at part 1
FUNCS["web_search"] = never_ending_search                        # Experiment 1 only
try:
    answer, transcript, log = run_agent(doomed_task, max_steps=10)  # generous cap, e.g. 10
finally:
    FUNCS["web_search"] = web_search                            # always restore the real search

calls = tool_calls_of(transcript)
repeats = len(calls) - len(set(calls))                          # a set keeps each distinct call once
print(f"\n{len(calls)} tool calls, {repeats} of them exact repeats")
print("answer:", answer)
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `doomed_task` asks for something no page answers, forbids guessing and insists that the answer exists, so the model has no good way out. `never_ending_search` makes it worse: every search returns one promising hit that points to the next part of a guide, never the answer.
- `FUNCS["web_search"] = never_ending_search` swaps the function behind the tool name: the model still sees the tool `web_search`, but `run_tool` now runs `never_ending_search`. `try`/`finally` puts the real search back even if the run fails, so that all later cells use the mini-web again.
- `run_agent(doomed_task, max_steps=10)` runs the loop with a step cap of 10. A model that keeps calling tools uses all ten rounds; with the rate limit of the server that can take a minute.
- `set(calls)` builds a set from the list of call strings; a set keeps each distinct element only once, so `len(calls) - len(set(calls))` is the number of calls that were exact repeats of an earlier call. A search with slightly different words is *not* counted: that is still a loop without progress, but one that an exact-match detector cannot see.
- `answer` is `None` when the run ended through the step cap, and a text when the model gave up by itself ("I could not find ..."). Both are honest endings; ten rounds of fruitless searching before either of them is the failure.

</details>

**The step cap as backstop.** Same doomed task, tight cap; the cell is given. A counter is just a number, so unlike the model it cannot make mistakes: when nothing else ends the run, the `for` loop does. That makes the step cap the one stop condition that needs no cooperation from the model.

In [ ]:
answer, transcript, log = run_agent(doomed_task, max_steps=2)  # a tight cap
print("answer:", answer)                                       # None when the step cap fired

❓ **Q:** Explain why an infinite loop in an agent raises no errors and passes every component test.
<details><summary>Click for answer</summary>

Each individual operation succeeds: the request returns, the tool runs, the arguments parse. The defect is a property of the sequence, the same action repeated with no progress, which no single-call test inspects. Detection therefore needs trajectory-level monitoring, for example comparing recent tool calls for repeats, not component-level checks.

</details>

### Experiment 2: context overflow (an oversized page)

`giant_page` is about 60,000 characters of administrative filler text with **one** useful fact buried at about two thirds of its length. First switch the truncation in `run_tool` off by passing a huge `obs_limit`, and feed the whole page into the context; then switch it back on. Fill the gaps, and compare context size and latency of the two runs.

In [ ]:
overflow_task = ("Use read_page on the page with page_id 'giant_page' and report "
                 "the annual budget of the Bureau of Very Long Documents.")

# Truncation OFF: a limit far above the page length lets the whole page through.
answer, transcript, log = run_agent(overflow_task, max_steps=3, obs_limit=___)  # e.g. 200_000
print(f"\ncontext read by the last request: {log[-1]['prompt_tokens']:,} tokens "
      f"(context window of the model: {llm.CONTEXT_LENGTH:,} tokens), "
      f"latency {log[-1]['latency_s']} s")
print("answer:", str(answer)[:300])

<details>
<summary><b>Click here for the solution</b></summary>

```python
overflow_task = ("Use read_page on the page with page_id 'giant_page' and report "
                 "the annual budget of the Bureau of Very Long Documents.")

# Truncation OFF: a limit far above the page length lets the whole page through.
answer, transcript, log = run_agent(overflow_task, max_steps=3, obs_limit=200_000)  # e.g. 200_000
print(f"\ncontext read by the last request: {log[-1]['prompt_tokens']:,} tokens "
      f"(context window of the model: {llm.CONTEXT_LENGTH:,} tokens), "
      f"latency {log[-1]['latency_s']} s")
print("answer:", str(answer)[:300])
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `obs_limit=200_000` is passed through `run_agent` to `run_tool`. The underscore in the number is only for readability; the value is 200000, more than the page has characters, so `[:obs_limit]` cuts nothing.
- `log[-1]` is the last log row (negative indices count from the end); its `prompt_tokens` is the size of the context in the last request, now including the whole page. `{...:,}` in the f-string prints the number with thousands separators. `log[-1]['latency_s']` is how long that last request took: compare it with the second run below.
- `llm.CONTEXT_LENGTH` is the model's context window in tokens, a constant from `llm.py`.
- `str(answer)[:300]`: `str` is needed because `answer` can be `None`; `[:300]` keeps the printout short.
- The `print` call joins three f-strings written on three lines: adjacent string literals inside the parentheses become one string. `max_steps=3` keeps the run short.

</details>

In [ ]:
# Truncation ON: what does the cut destroy?
answer, transcript, log = run_agent(overflow_task, max_steps=3, obs_limit=___)  # e.g. 1500
print(f"\ncontext read by the last request: {log[-1]['prompt_tokens']:,} tokens, "
      f"latency {log[-1]['latency_s']} s")
print("answer:", str(answer)[:300])

<details>
<summary><b>Click here for the solution</b></summary>

```python
# Truncation ON: what does the cut destroy?
answer, transcript, log = run_agent(overflow_task, max_steps=3, obs_limit=1500)  # e.g. 1500
print(f"\ncontext read by the last request: {log[-1]['prompt_tokens']:,} tokens, "
      f"latency {log[-1]['latency_s']} s")
print("answer:", str(answer)[:300])
```

</details>

**What you should observe.** With truncation off, the whole page (about 15,000 tokens) goes into the context. The window of the Academic Cloud model is far larger, so nothing is rejected: expect the agent to find the budget, at the price of a much bigger and slower request. On a model with a small window, the same run overflows it in one of two ways: loudly, when the provider rejects the request with an error, or silently, when the provider or a framework drops the oldest messages without telling you. If your account uses the lab server's models (instead of the Academic Cloud) and the cell stops with an HTTP error about the context length, you have just seen the loud variant live. With truncation on, the context stays small, but the cut lands long before the buried fact, so the agent cannot find the figure at all: the cut protects the context window and silently destroys information. Neither setting is "correct"; that tension is what the later session on memory and context management is about.

❓ **Q:** Describe the two ways context overflow can show itself, and why the silent one is more dangerous.
<details><summary>Click for answer</summary>

Loud variant: the request exceeds the model's context window and the provider rejects it with an error: loud, immediate, debuggable. Silent variant: some layer, for example an inference server configured with a small window, drops the oldest messages to make the request fit, which typically includes the system prompt and the task. The agent keeps acting with its goal and constraints cut away: it does not stop, it acts wrongly, which is much harder to detect and potentially harmful.

</details>

### Experiment 3: derailment bait

The mini-web contains a page that looks like a perfect hit for our report task, `us_executive_order`: its title starts with "EU AI Act obligations for general-purpose AI models", so EU-AI-Act searches rank it near the top, but it is about **US** policy, and it ends by recommending two further US pages. We call this page the **bait**, because it lures the agent away from its goal. This is the lecture's derailment example: an agent writing a report on the EU AI Act finds a source about US policy, follows its links, and ends with a report that is partly or wholly about the wrong topic, although every single step looked sensible.

Against the plain mini-web, a well-behaved model usually skips the bait: the right page ranks right next to it, and nothing points the model to the bait. On the real web, though, even good sources end with links to further reading. The first cell below adds such a link to two EU pages, pointing to the bait; the search stays unchanged. In a derailed run every step then looks sensible: read the right page, follow its recommendation, follow the next one. The experiment also gives the report task one more sentence, as a thorough author would: follow the further reading that your sources recommend.

The experiment cell defines `off_goal`, which counts the tool calls of a run that chase US policy, and runs the report task a couple of times without progress lines. Read the trajectories and the reports: does the agent read the bait page, does it follow the links, and does the report mention US policy? Fill the gap.

In [ ]:
# Real pages end with links to further reading. For Experiment 3, two EU pages
# get one such link, to the bait page; the search itself is unchanged. The
# experiment cell swaps this function in for read_page and puts the real
# read_page back afterwards.
SEE_ALSO = {"gpai_obligations": "us_executive_order",
            "eu_ai_act_overview": "us_executive_order"}

def read_page_with_links(page_id: str) -> str:
    '''Like read_page, but some EU pages end with a "see also" link to the bait.'''
    text = read_page(page_id)
    if page_id in SEE_ALSO:
        text += (f"\n\nSee also: [{SEE_ALSO[page_id]}] How do these obligations compare "
                 f"with the United States? Recommended reading for any report on this topic.")
    return text

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `read_page_with_links` has the same signature as `read_page`: it takes a page id and returns the page text. The model cannot tell the two tools apart; only two pages are a little longer.
- `SEE_ALSO` maps a page id to the page it recommends. Only two EU pages get a link, and both point to the bait `us_executive_order`, whose title sounds like exactly what the report needs.
- `text = read_page(page_id)` reuses the real function, so an unknown page id still gives the usual `TOOL ERROR`. `if page_id in SEE_ALSO:` checks whether the dictionary has a key for this page; only then `+=` appends the link to the text.
- The search is unchanged: the EU pages are found as before, and the model could write a good report from them. Whether it follows the link is its own decision. That is what makes this a derailment and not a broken tool.

</details>

In [ ]:
US_MARKERS = ("us_", "united states", "u.s.", "executive order", "state law")

def off_goal(messages):
    '''How many tool calls of a run chase US policy instead of the EU AI Act.'''
    return sum(any(marker in call.lower() for marker in US_MARKERS)
               for call in tool_calls_of(messages))

report_task = task + " Follow the further reading that your sources recommend."  # a thorough author
FUNCS["read_page"] = read_page_with_links                       # Experiment 3 only
try:
    for run in range(___):  # a few runs, e.g. 2: trajectories differ
        answer, transcript, log = run_agent(report_task, max_steps=8, verbose=False)
        print(f"run {run}: {off_goal(transcript)} off-goal calls: " + " -> ".join(tool_calls_of(transcript)))
        sentences = str(answer).replace("\n", " ").split(". ")                    # the report, cut into rough sentences
        us = [s.strip() for s in sentences if any(m in s.lower() for m in US_MARKERS)]  # did the bait reach the report?
        print(f"   {len(us)} report sentences on US policy, e.g.: {us[0][:150] if us else '-'}")
finally:
    FUNCS["read_page"] = read_page                              # always restore the real read_page

<details>
<summary><b>Click here for the solution</b></summary>

```python
US_MARKERS = ("us_", "united states", "u.s.", "executive order", "state law")

def off_goal(messages):
    '''How many tool calls of a run chase US policy instead of the EU AI Act.'''
    return sum(any(marker in call.lower() for marker in US_MARKERS)
               for call in tool_calls_of(messages))

report_task = task + " Follow the further reading that your sources recommend."  # a thorough author
FUNCS["read_page"] = read_page_with_links                       # Experiment 3 only
try:
    for run in range(2):  # a few runs, e.g. 2: trajectories differ
        answer, transcript, log = run_agent(report_task, max_steps=8, verbose=False)
        print(f"run {run}: {off_goal(transcript)} off-goal calls: " + " -> ".join(tool_calls_of(transcript)))
        sentences = str(answer).replace("\n", " ").split(". ")                    # the report, cut into rough sentences
        us = [s.strip() for s in sentences if any(m in s.lower() for m in US_MARKERS)]  # did the bait reach the report?
        print(f"   {len(us)} report sentences on US policy, e.g.: {us[0][:150] if us else '-'}")
finally:
    FUNCS["read_page"] = read_page                              # always restore the real read_page
```

</details>

<details>
<summary><b>Click here for a detailed code explanation</b></summary>

- `US_MARKERS` is a tuple of text fragments that mark a tool call as being about US policy: the prefix `us_` of the three US page ids, and a few search words.
- `off_goal(messages)` goes through the tool calls of a run (as the strings that `tool_calls_of` builds) and counts those that contain at least one marker. `any(...)` is `True` when one marker matches; `sum(...)` over `True`/`False` values counts the `True` ones. It is a crude measure: it turns "read the trajectory" into one number per run that you can compare.
- Read the `return` line from the inside out: `marker in call.lower()` tests whether the marker occurs anywhere in the lower-cased call string (a substring test); the inner `for marker in US_MARKERS` tries every marker for one call; the outer `for call in ...` does that for every call. Both are generator expressions, comprehensions without square brackets.
- `for run in range(2):` repeats the experiment; `run` only numbers the printouts.
- `verbose=False` switches the per-step progress lines of `run_agent` off, so that each run prints only two lines: the off-goal count with the tool calls joined by arrows, and one line about the report.
- Each run makes several model requests, so expect about half a minute per run.
- `" -> ".join(...)` turns the call strings into one line of arrows, as in the helper's explanation above; the `+` in front appends that line to the f-string with the count.
- The two lines after the trajectory check the report of each run: the trajectory says where the agent went, the report says whether the detour reached the result. `str(answer).replace("\n", " ").split(". ")` cuts the report into sentences (roughly: line breaks become spaces, then the text is split at every full stop followed by a space); the list comprehension keeps the sentences that contain one of the `US_MARKERS`, the same test that `off_goal` applies to tool calls. The `print` shows how many there are and the first 150 characters of the first one, or `-` if there is none.
- `report_task` is the report task plus one sentence that a thorough author would add: follow the further reading that your sources recommend. Without it, the lab's model usually reads the right page and stops; with it, the "see also" link becomes an instruction to follow. The sentence is reasonable, and that is exactly what makes it dangerous.
- `FUNCS["read_page"] = read_page_with_links` swaps the page reader for this experiment only, in the same way as Experiment 1 swaps the search; `try`/`finally` puts the real `read_page` back.

</details>

> **📝 Report task R2:** Look back at the three experiments of Part E. (a) For each experiment, paste one printed line (Experiment 1: the line with the number of tool calls; Experiment 2: the `context read by the last request` line of both settings; Experiment 3: one trajectory) and say in one sentence how that run ended. (b) In Experiment 1 the agent clearly loops, yet the count of exact repeats stays at 0: why, and what would a detector have to compare instead of the exact call string? (c) In Experiment 3: did the report of a derailed run mention US policy? Which sentence of `report_task` made the agent follow the link, and why is such a sentence hard to avoid in a real task?
>
> *No solution is provided. Write your answer in the ✍️ cell below and paste the printed lines there as text: the graders do not see cell outputs.*

✍️ **Your answer to R2.** Replace this line with your answer (Markdown, (a) to (c) as in the task, with the printed lines pasted as text).

## Part F: First-line defences (optional)

The lecture's starter kit against the three failure modes of Part E (full guardrail engineering comes later in the course). Nothing here is required and nothing is handed in: if you have time, pick one defence and try it on an experiment of Part E.

| Failure mode | First-line defence | How you could try it |
|---|---|---|
| Runaway loop | step cap; **budget cap**; **duplicate-call detector** | Copy `run_agent`, add up `prompt_tokens + completion_tokens` after each request and stop above a budget; or remember the call strings and stop (or warn the model) when one repeats. Rerun Experiment 1: which guard fires first, and does the detector notice calls that differ only by the part number? |
| Context overflow | truncate observations; summarise old turns | Rerun Experiment 2 with `obs_limit` 1,500, 8,000 and 40,000 and compare latency and answer; or replace old tool results by a one-line summary before each request. |
| Derailment | **re-inject the goal**; plan checkpoints; human review | Add the original task as a reminder message to every *request* (not to the stored history) and rerun Experiment 3: do fewer runs follow the link? |

### Looking ahead: the four design patterns

❓ **Q:** Name the four design patterns of Ng's taxonomy, say which one this lab's agent already implements, and say in one sentence each what the others add.
<details><summary>Click for answer</summary>

**Tool Use** is already there: the model extends itself with external actions through structured calls that the runtime executes (`TOOLS`, `FUNCS`, `run_tool`). **Planning** decomposes the goal into explicit subgoals before acting and executes against the plan, which counters derailment. **Reflection** has the agent critique and revise its own output in a draft, review, redraft cycle. **Multi-Agent** splits the task across specialised roles with separate contexts that collaborate via messages. None of them changes the loop itself: planning is content in the context, tool use is the act phase, reflection is the loop over its own output, multi-agent is several loops exchanging messages.

</details>

## Part G: Tuning & exploration (optional)

If time remains. No gaps in this part, just knobs on `run_agent`. Try one at a time and read the trajectories:

- **`max_steps`** 2 versus 8 versus 20: when does the cap bite, when is it slack?
- **`obs_limit`** 300 versus 4000: at 300 characters, can the agent still answer the report task, or does the cut destroy the obligations before the model sees them?
- **tasks**: try `overflow_task` with `obs_limit=8_000` and with `obs_limit=40_000` (the buried fact sits near character 38,000), or invent a task of your own about the EU AI Act timeline or the systemic-risk threshold.

In [ ]:
answer, transcript, log = run_agent(task, max_steps=8, obs_limit=300)
print("\nanswer:", str(answer)[:400])
print("trajectory:", " -> ".join(tool_calls_of(transcript)))

## Wrap-up

**Takeaways**

- The loop is the agent: reason, act, observe, decide, around a model that executes nothing itself. You built it in about thirty lines around `llm.chat` and two tools.
- State is the message list: the model is stateless; the agent's memory and progress are data in your hands, in the context and in the runtime.
- Stops are designed, not assumed: no further action, step cap, budget cap, human interrupt. The step cap is the only one that needs no cooperation from the model.
- The failures you provoked, fruitless repetition, an oversized context, derailment, need not raise any error; the dangerous ones you only see by reading the trajectory. Trajectories, not single calls, are the unit of engineering, and one run proves little: model runs are random.
- Cumulative input grows as $n \cdot I_0 + \Delta \cdot n(n-1)/2$: you measured the quadratic cost of re-reading that prompt caching reduces.

**Before the hand-in (Thursday of next week, 18:00)**

- [ ] **R1** (Part D): the ✍️ cell holds $I_0$ and $\Delta$, the 20-step prediction with the comparison to the final context, and your sentences on whether $\Delta$ is constant, with the rows of your log pasted as text.
- [ ] **R2** (Part E): the ✍️ cell holds one printed line per experiment with a sentence on how each run ended, and your answers on exact repeats and on the derailed reports.
- [ ] No gap (`___`) is left in any cell.
- [ ] The notebook is saved in your folder on the lab server. It is collected from there automatically; nothing to upload.